![MOMENTUM · Hackatón de DRL · 03 Wrappers](../img/banner-e03.jpg)

# **3. _Wrappers_**

Los **_wrappers_** son una herramienta de `gymnasium` especialmente útil. Se trata de clases que permiten modificar el comportamiento de un entorno sin tener que modificar el código original del mismo.

Existen tres tipos de _wrappers_ principales:

1. _Wrappers_ de **acciones**: modifican las acciones antes de enviarlas al entorno.
2. _Wrappers_ de **observaciones**: modifican la observación / estado que percibe al agente.
3. _Wrappers_ de **recompensas**: modifican la recompensa que recibe el agente.

También existen otros tipos de _wrappers_ auxiliares destinados a registrar estadísticas, monitorizar el entrenamiento, etc. Si bien podemos implementarlos manualmente, `gymnasium` ofrece una serie de _wrappers_ ya implementados que pueden consultarse [aquí](https://gymnasium.farama.org/api/wrappers/table/).

Veamos en qué consisten. Comenzamos importando las librerías necesarias:

In [ ]:
import gymnasium as gym
import highway_env

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def show_state(env):
    """
    Muestra el estado actual del entorno.
    """
    img = env.render()
    clear_output(wait=True)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

## **3.1. _Wrappers_ de acciones**

Estos _wrappers_ permiten modificar las **acciones** que el agente ejecuta a través de `step`.

Por ejemplo, en el caso de `highway-env`, el espacio de acciones original es:

- `0`: cambiar al carril izquierdo.
- `1`: mantener velocidad y carril actuales.
- `2`: cambiar al carril derecho.
- `3`: acelerar.
- `4`: reducir velocidad.

Supongamos que queremos impedir que el agente cambie de carril. Esto supondría evitar que el agente realice las acciones `0` y `2`, y que estas se reemplacen por la acción `1` (mantener carril). Para lograr esto, podemos definir el _wrapper_ `NonChangeWrapper`:

In [ ]:
class NonChangeWrapper(gym.ActionWrapper):
    """
    Clase `NonChangeWrapper`. Implementa un wrapper que imposibilita elegir las acciones 0 (cambiar al carril izquierdo) y 2 (cambiar al carril derecho).
    Dichas acciones se sustituyen por 1 (mantener carril).
    """

    # Aqui se define como se manipulan las acciones del agente
    def action(self, action):
        if action == 0 or action == 2:
            action = 1
        return action

Para aplicar el _wrapper_ a nuestro entorno, es necesario "envolverlo", tal que así:

In [ ]:
# Creamos el entorno
env = gym.make("highway-v0", render_mode="rgb_array")

# Lo "envolvemos"
env = NonChangeWrapper(env)

# Mostramos el estado original
env.reset()
show_state(env)

Veamos ahora cómo funciona el _wrapper_:

In [ ]:
# Ejecutamos la accion 0 (cambiar al carril izquierdo)
env.step(0)
show_state(env)

Observamos cómo el cambio a la izquierda no se produce. 

Probamos lo mismo con el cambio al carril derecho:

In [ ]:
# Ejecutamos la accion 2 (cambiar al carril derecho)
env.step(2)
show_state(env)

De nuevo, la acción no tiene efecto sobre el vehículo, manteniéndose en el mismo carril.

Si bien, a priori, se trata de un _wrapper_ de poca utilidad, nos sirve para entender en qué consisten estos _wrappers_ y cómo utilizarlos.

Existen otros `ActionWrapper` ya implementados en `gymnasium`, los cuales pueden consultarse [aquí](https://gymnasium.farama.org/api/wrappers/action_wrappers/#). Algunos de estos son especialmente útiles para entrenar agentes, permitiendo la normalización de las acciones o el `clipping` (mantener las acciones dentro de un rango predefinido).

# **3.2. _Wrappers_ de observaciones**

Estos _wrappers_ permiten modificar el **estado** u **observación** percibido por el agente.

De nuevo, veamos un ejemplo donde implementamos un _wrapper_ de observación de forma manual. Recordemos qué información contiene el estado del agente en `highway-v0`:

In [ ]:
env = gym.make("highway-v0", render_mode="rgb_array")

# Obtenemos el estado inicial
state, _ = env.reset()

state

Dicha matriz representa un estado que contiene información sobre la posición y velocidad ($m/s$) de los vehículos en pantalla (incluido el vehículo controlado por el agente):

<div style="text-align: left;">

| Vehículo   | Posición X | Posición Y | Velocidad X | Velocidad Y |
|------------|-----------:|-----------:|------------:|------------:|
| agente     | 5.0        | 4.0        | 15.0        | 0           |
| vehículo 1 | -10.0      | 4.0        | 12.0        | 0           |
| vehículo 2 | 13.0       | 8.0        | 13.5        | 0           |
| …          | …          | …          | …           | …           |
| vehículo V | 22.2       | 10.5       | 18.0        | 0.5         |

</div>

A modo de ejemplo, vamos a definir el _wrapper_ `KmHourWrapper`, que convierte las velocidades a kilómetros por hora:

In [ ]:
class KmHourWrapper(gym.ObservationWrapper):
    """
    Clase `KmHourWrapper`. Implementa un wrapper que convierte la velocidad en km/h.
    """

    # Aqui se define como se manipulan las observaciones del agente
    def observation(self, observation):
        # Conversion de las columnas 3 y 4 a km/h (multiplicando por 3.6)
        observation[:, 2:4] *= 3.6
        return observation
        

Ahora veamos cómo se aplica dicho _wrapper_:

In [ ]:
env = gym.make("highway-v0", render_mode="rgb_array")

# Estado inicial sin wrapper:
state_original, _ = env.reset()

env = KmHourWrapper(env)

# Estado inicial tras aplicar el wrapper:
state_wrapper, _ = env.reset()

print("Estado inicial original (velocidades en m/s): \n", state_original)
print("Estado inicial aplicando el wrapper (velocidades en km/h): \n", state_wrapper)

De nuevo, se trata de un ejemplo de uso que no es necesariamente útil para controlar mejor nuestro vehículo. No obstante, existen algunos _wrappers_ preimplementados que merecen cierta atención. Algunas de las utilidades más comunes son: normalizar las variables percibidas, concatenar múltiples observaciones consecutivas o añadir/eliminar variables observadas. 

La lista de _wrappers_ de observación ya implementados en `gymnasium` puede consultarse [aquí](https://gymnasium.farama.org/api/wrappers/observation_wrappers/).

## **3.3. _Wrappers_ de recompensa**

Finalmente, veamos un ejemplo de _wrapper_ de **recompensa**. En este caso, utilizaremos un _wrapper_ preimplementado en `gymnasium`: [`NormalizeReward`](https://gymnasium.farama.org/api/wrappers/reward_wrappers/#gymnasium.wrappers.NormalizeReward), que normaliza las recompensas siguiendo el método de la media móvil. Se trata de un factor especialmente relevante para facilitar el entrenamiento de agentes basados en redes neuronales.

Para utilizar este _wrapper_, importamos `NormalizeReward` y "envolvemos" nuestro entorno:

In [ ]:
from gymnasium.wrappers import NormalizeReward

env = gym.make("highway-v0", render_mode="rgb_array")

# Aplicamos el wrapper
env = NormalizeReward(env)

Puedes consultar otros _wrappers_ de recompensa preimplementados [aquí](https://gymnasium.farama.org/api/wrappers/reward_wrappers/#reward-wrappers).

## **3.4. Otros _wrappers_**

Existen múltiples _wrappers_ ya implementados en `gymnasium` que pueden ser de utilidad para facilitar la monitorización y entrenamiento de los agentes.

Por ejemplo:

- [`AutoReset`](https://gymnasium.farama.org/api/wrappers/misc_wrappers/#gymnasium.wrappers.Autoreset): reinicia automáticamente el entorno cuando `terminated` o `truncated` es `True`.
- [`RecordVideo`](https://gymnasium.farama.org/api/wrappers/misc_wrappers/#gymnasium.wrappers.RecordVideo): permite la grabación de vídeos del entorno utilizando el método `render`.
- [`TimeLimit`](https://gymnasium.farama.org/api/wrappers/misc_wrappers/#gymnasium.wrappers.TimeLimit): define un límite de tiempo a partir del cual un episodio es truncado.

Siguiendo la API de `gymnasium`, la librería `stable-baselines3` también ofrece una serie de _wrappers_ preimplementados que facilitan la monitorización de los agentes. Véase:

- [`Monitor`](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html#stable_baselines3.common.monitor.Monitor): recopila información sobre recompensa, longitud y tiempo de ejecución de cada episodio.
- [`ResultsWriter`](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html#stable_baselines3.common.monitor.ResultsWriter): almacena los datos de `Monitor` en un fichero.


## **3.5. Recomendaciones finales**

Una vez completado este cuaderno, contamos con una perspectiva general de las herramientas ofrecidas por `gymnasium` y `stable-baselines3`. A partir de aquí, se recomienda experimentar con diferentes configuraciones, _wrappers_, _callbacks_ y agentes en el entorno `highway-v0`, tratando de obtener un buen rendimiento e incluso generalizar a otros entornos similares.

Como material adicional, se recomienda consultar la documentación oficial de [`gymnasium`](https://gymnasium.farama.org/) y [`stable-baselines3`](https://stable-baselines3.readthedocs.io/en/master/index.html). Otros recursos de interés son: 

- [`sb3-contrib`](https://github.com/Stable-Baselines-Team/stable-baselines3-contrib): proyecto que incluye algoritmos adicionales a los ofrecidos por `stable-baselines3`.
- [`sbx`](https://github.com/araffin/sbx): repositorio con implementaciones más rápidas y eficientes de los algoritmos de `stable-baselines3`.
- [_Reinforcement learning tips and tricks_](https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html): recomendaciones para el entrenamiento y evaluación de agentes de DRL.
- [_RL Baselines3 Zoo_](https://github.com/DLR-RM/rl-baselines3-zoo): _framework_ para optimización de hiperparámetros.
- [Integración de _Tensorboard_](https://stable-baselines3.readthedocs.io/en/master/guide/tensorboard.html) en el entrenamiento de agentes.

A partir de aquí, una buena aproximación inicial al problema propuesto sería la siguiente:

1. Elegir algoritmos robustos a hiperparámetros y que se encuentren en el estado del arte. Puedes probar un algoritmo _on-policy_ (ej. `PPO`) y otro _off-policy_ (ej. `SAC`).
2. Aplicar _wrappers_ de normalización sobre las acciones, observaciones y recompensas. Considerar si se añade información adicional a las observaciones o si se aplica [_stacking_](https://gymnasium.farama.org/api/wrappers/observation_wrappers/#gymnasium.wrappers.FrameStackObservation).
3. Definir _callbacks_ para evaluación periódica en un entorno de evaluación aislado.
4. Entrenar al agente durante suficientes _timesteps_. Recuerda utilizar `optuna` para optimización de hiperparámetros, intentando explorar diferentes valores en aquellos que consideres más relevantes.
5. Monitoriza el entrenamiento, usando tanto `Monitor` como `tensorboard`. De esta forma podrás saber si la función de recompensa converge, o si es necesario entrenar durante más tiempo.
6. Evalúa y reentrena si es necesario. Recuerda guardar siempre tus mejores modelos.

**¡Buena suerte! :-)**
